# Pumpkinmeter: Collaborative Movie Recommendation System using Apache Spark

## Project Overview

Ripe Pumpkins, a movie review aggregation startup, aims to develop a collaborative recommendation system called *Pumpkinmeter* to improve personalized movie recommendations for users. Inspired by the success of recommendation systems used by major streaming platforms, the company would like to explore how collaborative filtering can identify individual customer preferences and improve customer engagement and retention.

This project develops a movie recommendation engine using Apache Spark MLlib’s Alternating Least Squares (ALS) collaborative filtering algorithm on the MovieLens dataset provided by GroupLens Research.

The recommendation system generates personalized movie recommendations based on user rating behavior and evaluates how different filtering thresholds impact recommendation quality and diversity.


# Project Objective

The main objective of this project is to build a scalable collaborative filtering recommendation engine using Apache Spark and the ALS algorithm to generate personalized movie recommendations for users.

The project specifically aims to:

- Analyze user movie rating behavior using collaborative filtering
- Build a recommendation engine using Spark MLlib ALS
- Generate personalized movie recommendations for new users
- Compare recommendation performance under different filtering scenarios
- Explore how recommendation systems may improve customer satisfaction and retention for streaming platforms


# Data Source

This project uses the MovieLens dataset collected and published by GroupLens Research.

Dataset Source:
https://grouplens.org/datasets/movielens/


# Source Article Reference

This project implementation was guided by the following Apache Spark collaborative filtering tutorial:

Jadianes, Alejandro. *Building a Recommendation Engine with Apache Spark*. Codementor.  
Source Article: https://www.codementor.io/@jadianes/building-a-recommender-with-apache-spark-python-example-app-part1-du1083qbw

The source article provided the foundational workflow for:
- loading and preprocessing MovieLens datasets
- creating Spark RDDs
- implementing collaborative filtering using Apache Spark MLlib
- training ALS recommendation models
- tuning ALS parameters using RMSE
- generating personalized movie recommendations

This project extends the source article by applying the recommendation workflow to the FULL MovieLens dataset and implementing the four required recommendation scenarios specified in the Data Translation Challenge (DTC) project instructions.



# Use of Small and Full MovieLens Datasets

The smaller MovieLens dataset (`ml-latest-small`) is initially used for testing, ALS parameter tuning, and model evaluation because it is computationally efficient and suitable for experimentation. This includes tasks such as train-validation-test splitting, RMSE evaluation, and selecting appropriate ALS model parameters.

After selecting suitable ALS parameters, the final collaborative filtering recommendation system is trained using the FULL MovieLens dataset (`ml-latest`). The full dataset contains significantly larger user-movie interactions and is used for generating personalized movie recommendations under the required recommendation scenarios.

Using the full dataset allows the recommendation engine to better capture user preference patterns and produce more realistic large-scale collaborative filtering recommendations.

In [1]:
complete_dataset_url = \
'http://files.grouplens.org/datasets/movielens/ml-latest.zip'

small_dataset_url = \
'http://files.grouplens.org/datasets/movielens/ml-latest-small.zip'

In [2]:
# Import required libraries

import os
import urllib
import zipfile
from pyspark.sql import SparkSession

In [3]:
# Create Spark session

spark = SparkSession.builder \
    .appName("Pumpkinmeter Recommendation System") \
    .getOrCreate()

# Create SparkContext

sc = spark.sparkContext

In [4]:
# Extract small MovieLens dataset

with zipfile.ZipFile(
    "ml-latest-small.zip",
    "r"
) as z:
    
    z.extractall()

In [5]:
# Extract full MovieLens dataset

with zipfile.ZipFile(
    "ml-latest.zip",
    "r"
) as z:
    
    z.extractall()

# Loading and Parsing MovieLens Datasets

The MovieLens datasets contain multiple CSV files including ratings, movies, tags, and links.

The ratings dataset (`ratings.csv`) contains:
- userId
- movieId
- rating
- timestamp

The movies dataset (`movies.csv`) contains:
- movieId
- movie title
- genres

For collaborative filtering recommendation modeling, the project mainly uses:
- ratings data
- movie titles

The timestamp and genres fields are not required for ALS collaborative filtering and are therefore excluded during parsing.

The small MovieLens dataset is first loaded and parsed for ALS testing and parameter evaluation.

## Loading Small Ratings Dataset

The ratings dataset is loaded as a Spark RDD using `sc.textFile()`. 

The header row is removed before parsing the ratings into tuples containing:
(UserID, MovieID, Rating)

In [6]:
# Define path for small ratings dataset

small_ratings_file = os.path.join(
    'ml-latest-small',
    'ratings.csv'
)

In [7]:
# Load raw ratings data

small_ratings_raw_data = sc.textFile(
    small_ratings_file
)

# Extract header

small_ratings_raw_data_header = \
small_ratings_raw_data.take(1)[0]

In [8]:
# Parse ratings dataset into RDD

small_ratings_data = small_ratings_raw_data \
    .filter(
        lambda line:
        line != small_ratings_raw_data_header
    ) \
    .map(
        lambda line:
        line.split(",")
    ) \
    .map(
        lambda tokens:
        (
            tokens[0],
            tokens[1],
            tokens[2]
        )
    ) \
    .cache()

In [9]:
# Display sample records

small_ratings_data.take(5)

[('1', '31', '2.5'),
 ('1', '1029', '3.0'),
 ('1', '1061', '3.0'),
 ('1', '1129', '2.0'),
 ('1', '1172', '4.0')]

## Loading Small Movies Dataset

The movies dataset is loaded into a Spark RDD and parsed into tuples containing:
(MovieID, Movie Title)

Genres are excluded because they are not directly used in ALS collaborative filtering recommendations.

In [10]:
# Define path for small movies dataset

small_movies_file = os.path.join(
    'ml-latest-small',
    'movies.csv'
)

In [11]:
# Load raw movies data

small_movies_raw_data = sc.textFile(
    small_movies_file
)

# Extract header

small_movies_raw_data_header = \
small_movies_raw_data.take(1)[0]

In [12]:
# Parse movies dataset into RDD

small_movies_data = small_movies_raw_data \
    .filter(
        lambda line:
        line != small_movies_raw_data_header
    ) \
    .map(
        lambda line:
        line.split(",")
    ) \
    .map(
        lambda tokens:
        (
            tokens[0],
            tokens[1]
        )
    ) \
    .cache()

In [13]:
# Display sample movie records

small_movies_data.take(5)

[('1', 'Toy Story (1995)'),
 ('2', 'Jumanji (1995)'),
 ('3', 'Grumpier Old Men (1995)'),
 ('4', 'Waiting to Exhale (1995)'),
 ('5', 'Father of the Bride Part II (1995)')]

# Collaborative Filtering using ALS

In this step, the collaborative filtering recommendation model is prepared using Apache Spark MLlib’s Alternating Least Squares (ALS) algorithm.

The small MovieLens dataset is first converted into ALS rating format and then divided into training, validation, and test datasets. This allows the recommendation model to be trained and evaluated using different ALS parameter combinations before applying the final recommendation system to the FULL MovieLens dataset.

The purpose of this step is to identify suitable ALS model parameters and evaluate recommendation performance using RMSE before generating personalized movie recommendations on the large-scale dataset.

In [14]:
# Import ALS libraries

from pyspark.mllib.recommendation import ALS
from pyspark.mllib.recommendation import Rating

## Converting Ratings into ALS Format

The ratings dataset is converted into Spark MLlib ALS `Rating` objects.

Each rating is stored in the following format:

(UserID, MovieID, Rating)

This format is required by the ALS collaborative filtering algorithm before training recommendation models.

In [15]:
# Convert ratings into ALS Rating format

small_ratings_data = small_ratings_data.map(
    lambda l: Rating(
        int(l[0]),
        int(l[1]),
        float(l[2])
    )
)

In [16]:
# Display sample ALS ratings

small_ratings_data.take(5)

[Rating(user=1, product=31, rating=2.5),
 Rating(user=1, product=1029, rating=3.0),
 Rating(user=1, product=1061, rating=3.0),
 Rating(user=1, product=1129, rating=2.0),
 Rating(user=1, product=1172, rating=4.0)]

## Evaluating the Final ALS Model

After selecting the best ALS rank using the validation dataset, the final ALS recommendation model is trained and evaluated on the test dataset.

The recommendation performance is measured using Root Mean Square Error (RMSE). RMSE compares the predicted movie ratings generated by the ALS model with the actual movie ratings in the test dataset.

Lower RMSE values indicate better recommendation accuracy and stronger collaborative filtering performance.

In [17]:
# Split small dataset into
# training, validation, and test datasets

training_RDD, validation_RDD, test_RDD = \
small_ratings_data.randomSplit(
    [6, 2, 2],
    seed=0
)

In [18]:
# Prepare validation and test datasets
# for ALS prediction

validation_for_predict_RDD = validation_RDD.map(
    lambda x: (x[0], x[1])
)

test_for_predict_RDD = test_RDD.map(
    lambda x: (x[0], x[1])
)

# Using the FULL Dataset to Build the Final Recommendation Model

After evaluating the ALS collaborative filtering model using the small MovieLens dataset, the final recommendation engine is built using the FULL MovieLens dataset.

The complete dataset contains significantly larger user-movie interactions, allowing the recommendation model to better capture user preference patterns and generate large-scale personalized movie recommendations.

Following the same preprocessing workflow used for the small dataset, the ratings data is loaded, parsed, converted into ALS rating format, and cached for recommendation modeling.

In [19]:
# Define path for full ratings dataset

complete_ratings_file = os.path.join(
    'ml-latest',
    'ratings.csv'
)

In [20]:
# Load raw full ratings data

complete_ratings_raw_data = sc.textFile(
    complete_ratings_file
)

# Extract header

complete_ratings_raw_data_header = \
complete_ratings_raw_data.take(1)[0]

In [21]:
# Parse full ratings dataset into ALS format

complete_ratings_data = \
complete_ratings_raw_data.filter(
    lambda line:
    line != complete_ratings_raw_data_header
).map(
    lambda line:
    line.split(",")
).map(
    lambda tokens:
    Rating(
        int(tokens[0]),
        int(tokens[1]),
        float(tokens[2])
    )
).cache()

In [22]:
# Display total ratings count

print(
    "There are %s ratings in the complete dataset"
    % (
        complete_ratings_data.count()
    )
)

There are 33832162 ratings in the complete dataset


# Creating Recommendation Scenarios

To evaluate how filtering thresholds affect personalized movie recommendations, two recommendation scenarios are created using the FULL MovieLens dataset.

Movies with very few ratings may introduce noise and reduce recommendation reliability. Therefore, filtering thresholds are applied before training the ALS recommendation models.

Two filtering scenarios are used in this project:

## Scenario 1
Include movies with at least 25 ratings.

## Scenario 2
Include movies with at least 100 ratings.

These scenarios allow comparison between more diverse recommendations and more popularity-focused recommendations.

In [23]:
# Count number of ratings for each movie

movie_rating_counts = complete_ratings_data.map(
    lambda x:
    (x[1], 1)
).reduceByKey(
    lambda x, y:
    x + y
)

In [24]:
# Filter movies with at least 25 ratings

movies_25 = movie_rating_counts.filter(
    lambda x:
    x[1] >= 25
)

In [25]:
# Create Scenario 1 dataset

ratings_25 = complete_ratings_data.map(
    lambda x:
    (x[1], x)
).join(
    movies_25
).map(
    lambda x:
    x[1][0]
).cache()

In [26]:
# Display Scenario 1 ratings count

print(
    "Scenario 1 Ratings Count:",
    ratings_25.count()
)

Scenario 1 Ratings Count: 33513684


## Scenario 2: Movies with 100+ Ratings

In Scenario 2, movies with fewer than 100 ratings are removed from the recommendation dataset.

This stricter filtering threshold reduces sparsity and focuses the recommendation model on more frequently rated and widely recognized movies. The purpose of this scenario is to compare how stronger filtering conditions affect personalized recommendation quality and diversity.

In [27]:
# Filter movies with at least 100 ratings

movies_100 = movie_rating_counts.filter(
    lambda x:
    x[1] >= 100
)

In [28]:
# Create Scenario 2 dataset

ratings_100 = complete_ratings_data.map(
    lambda x:
    (x[1], x)
).join(
    movies_100
).map(
    lambda x:
    x[1][0]
).cache()

In [29]:
# Display Scenario 2 ratings count

print(
    "Scenario 2 Ratings Count:",
    ratings_100.count()
)

Scenario 2 Ratings Count: 33060369


# Adding New User Ratings

To evaluate the collaborative filtering recommendation engine, two new users are added to the recommendation system. The two users represent myself and a friend/family member, as required in the project instructions. Each user provides ratings for 10 movies based on personal viewing preferences.

These new ratings simulate how real users interact with a movie recommendation platform and allow the ALS recommendation model to generate personalized movie recommendations.

Two different user preference profiles are created in this project:

- User 1 (Myself): Primarily interested in action, science fiction, adventure, and drama movies.
- User 2 (Friend/Family Member): Primarily interested in animation, fantasy, family, romance, and comedy movies.

Before creating the rating lists, movie titles are searched in the MovieLens dataset to identify the correct movie IDs required by the ALS recommendation model.

The new user ratings will later be combined with both Scenario 1 (25+ ratings) and Scenario 2 (100+ ratings) datasets to generate top 15 personalized movie recommendations for each user.

## Loading Movies Dataset

The movies dataset is loaded to retrieve movie titles and movie IDs required for creating new user rating profiles for the ALS recommendation model.

In [30]:
# Load movies dataset

movies = spark.read.csv(
    "ml-latest/movies.csv",
    header=True,
    inferSchema=True
)

In [31]:
# Display sample movie records

movies.show(5, truncate=False)

+-------+----------------------------------+-------------------------------------------+
|movieId|title                             |genres                                     |
+-------+----------------------------------+-------------------------------------------+
|1      |Toy Story (1995)                  |Adventure|Animation|Children|Comedy|Fantasy|
|2      |Jumanji (1995)                    |Adventure|Children|Fantasy                 |
|3      |Grumpier Old Men (1995)           |Comedy|Romance                             |
|4      |Waiting to Exhale (1995)          |Comedy|Drama|Romance                       |
|5      |Father of the Bride Part II (1995)|Comedy                                     |
+-------+----------------------------------+-------------------------------------------+
only showing top 5 rows



## Identifying Movie IDs for User 1

Before creating the rating list for User 1, movie titles are searched in the MovieLens dataset to identify their corresponding movie IDs.

The ALS recommendation model works using numerical movie identifiers rather than movie titles directly.

In [32]:
from pyspark.sql.functions import col

In [33]:
# List of movies selected by User 1

user1_movies = [
    "Toy Story",
    "Matrix",
    "Titanic",
    "Inception",
    "Avengers",
    "Batman",
    "Interstellar",
    "Jurassic Park",
    "Star Wars",
    "Fight Club"
]

# Search movie IDs for each movie

for movie in user1_movies:
    
    print(f"\nSearching for: {movie}")
    
    movies.filter(
        col("title").contains(movie)
    ).show(truncate=False)


Searching for: Toy Story
+-------+-----------------------------------------+------------------------------------------------+
|movieId|title                                    |genres                                          |
+-------+-----------------------------------------+------------------------------------------------+
|1      |Toy Story (1995)                         |Adventure|Animation|Children|Comedy|Fantasy     |
|3114   |Toy Story 2 (1999)                       |Adventure|Animation|Children|Comedy|Fantasy     |
|78499  |Toy Story 3 (2010)                       |Adventure|Animation|Children|Comedy|Fantasy|IMAX|
|106022 |Toy Story of Terror (2013)               |Animation|Children|Comedy                       |
|115875 |Toy Story Toons: Hawaiian Vacation (2011)|Adventure|Animation|Children|Comedy|Fantasy     |
|115879 |Toy Story Toons: Small Fry (2011)        |Adventure|Animation|Children|Comedy|Fantasy     |
|120468 |Toy Story Toons: Partysaurus Rex (2012)  |Animation|Chil

+-------+--------------------------------------------------------+--------------------------------------+
|movieId|title                                                   |genres                                |
+-------+--------------------------------------------------------+--------------------------------------+
|153    |Batman Forever (1995)                                   |Action|Adventure|Comedy|Crime         |
|592    |Batman (1989)                                           |Action|Crime|Thriller                 |
|1377   |Batman Returns (1992)                                   |Action|Crime                          |
|1562   |Batman & Robin (1997)                                   |Action|Adventure|Fantasy|Thriller     |
|3213   |Batman: Mask of the Phantasm (1993)                     |Animation|Children                    |
|26152  |Batman (1966)                                           |Action|Adventure|Comedy               |
|33794  |Batman Begins (2005)                 

## Creating User 1 Ratings

After identifying the correct movie IDs, User 1 movie ratings are manually created using the format:

(userId, movieId, rating)

These ratings represent User 1’s personal movie preferences and will later be added to the recommendation datasets for personalized recommendation generation.

In [34]:
# User 1 movie ratings
# Format: (userId, movieId, rating)

user1_ratings = [

    (999999, 1, 5.0),        # Toy Story
    (999999, 2571, 5.0),     # Matrix
    (999999, 1721, 4.5),     # Titanic
    (999999, 79132, 5.0),    # Inception
    (999999, 89745, 4.5),    # Avengers
    (999999, 33794, 4.0),    # Batman Begins
    (999999, 109487, 5.0),   # Interstellar
    (999999, 480, 4.5),      # Jurassic Park
    (999999, 260, 5.0),      # Star Wars
    (999999, 2959, 4.5)      # Fight Club

]

## Identifying Movie IDs for User 2

Movie titles selected by User 2 are searched in the MovieLens dataset to identify the correct movie IDs required for creating personalized ratings for the ALS recommendation model.

In [35]:
# List of movies selected by User 2

user2_movies = [
    "Frozen",
    "Finding Nemo",
    "Avatar",
    "Toy Story 3",
    "Lord of the Rings",
    "Shrek",
    "Incredibles",
    "Harry Potter",
    "Moana",
    "Lion King"
]

# Search movie IDs for each movie

for movie in user2_movies:
    
    print(f"\nSearching for: {movie}")
    
    movies.filter(
        col("title").contains(movie)
    ).show(truncate=False)


Searching for: Frozen
+-------+----------------------------------------+--------------------------------------------------+
|movieId|title                                   |genres                                            |
+-------+----------------------------------------+--------------------------------------------------+
|31785  |Frozen Land (Paha maa) (2005)           |Drama                                             |
|60930  |Frozen City (Valkoinen kaupunki) (2006) |Drama                                             |
|60943  |Frozen River (2008)                     |Drama                                             |
|69981  |Winter of Frozen Dreams (2009)          |Crime|Thriller                                    |
|75395  |Frozen (2010)                           |Drama|Horror|Thriller                             |
|82452  |Frozen (2007)                           |Drama                                             |
|83835  |Frozen North, The (1922)                |Comedy   

+-------+------------+-------------------------------------------+
|movieId|title       |genres                                     |
+-------+------------+-------------------------------------------+
|73141  |Moana (1926)|Documentary                                |
|166461 |Moana (2016)|Adventure|Animation|Children|Comedy|Fantasy|
|211209 |Moana (2009)|Drama                                      |
+-------+------------+-------------------------------------------+


Searching for: Lion King
+-------+---------------------+-----------------------------------------------+
|movieId|title                |genres                                         |
+-------+---------------------+-----------------------------------------------+
|364    |Lion King, The (1994)|Adventure|Animation|Children|Drama|Musical|IMAX|
|203222 |The Lion King (2019) |Adventure|Animation|Children|Drama             |
+-------+---------------------+-----------------------------------------------+



# Creating User 2 Ratings

After identifying the correct movie IDs, User 2 movie ratings are manually created using the format:

(userId, movieId, rating)

These ratings represent User 2’s personal movie preferences and will later be added to the recommendation datasets for personalized recommendation generation.

In [36]:
# User 2 movie ratings
# Format: (userId, movieId, rating)

user2_ratings = [

    (999998, 4886, 5.0),      # Monsters, Inc.
    (999998, 106696, 4.5),    # Frozen
    (999998, 72998, 4.0),     # Avatar
    (999998, 78499, 5.0),     # Toy Story 3
    (999998, 5952, 4.5),      # Lord of the Rings: The Two Towers
    (999998, 4306, 4.0),      # Shrek
    (999998, 6377, 5.0),      # Finding Nemo
    (999998, 8961, 4.5),      # Incredibles
    (999998, 7153, 4.0),      # Lord of the Rings: Return of the King
    (999998, 4993, 5.0)       # Lord of the Rings: Fellowship of the Ring

]

## Converting New User Ratings into RDDs

The manually created ratings for User 1 and User 2 are converted into Spark RDDs so they can be combined with the Scenario 1 and Scenario 2 rating datasets.

In [37]:
# Convert User 1 ratings into ALS Rating format

user1_ratings_RDD = sc.parallelize(user1_ratings).map(
    lambda x: Rating(x[0], x[1], x[2])
)

# Convert User 2 ratings into ALS Rating format

user2_ratings_RDD = sc.parallelize(user2_ratings).map(
    lambda x: Rating(x[0], x[1], x[2])
)

In [38]:
# Convert Scenario 1 ratings into ALS Rating format

ratings_25 = ratings_25.map(
    lambda x: Rating(
        int(x[0]),
        int(x[1]),
        float(x[2])
    )
)

# Convert Scenario 2 ratings into ALS Rating format

ratings_100 = ratings_100.map(
    lambda x: Rating(
        int(x[0]),
        int(x[1]),
        float(x[2])
    )
)

## Combining New User Ratings with Scenario Datasets

After creating the personalized movie ratings for both users, the new ratings are combined with the two recommendation scenario datasets using Spark’s `union()` transformation.

The new user ratings are combined separately with both filtering scenarios:

### Scenario 1
Movies with at least 25 ratings are included in the recommendation dataset.

### Scenario 2
Movies with at least 100 ratings are included in the recommendation dataset.

Combining the datasets creates four separate recommendation cases required in the project:

- **User 1 – Scenario 1**  
  User 1 ratings combined with movies having 25 or more ratings.

- **User 1 – Scenario 2**  
  User 1 ratings combined with movies having 100 or more ratings.

- **User 2 – Scenario 1**  
  User 2 ratings combined with movies having 25 or more ratings.

- **User 2 – Scenario 2**  
  User 2 ratings combined with movies having 100 or more ratings.

In [39]:
# Create four required recommendation datasets

scenario1_user1 = ratings_25.union(user1_ratings_RDD)
scenario2_user1 = ratings_100.union(user1_ratings_RDD)

scenario1_user2 = ratings_25.union(user2_ratings_RDD)
scenario2_user2 = ratings_100.union(user2_ratings_RDD)

print("User 1 - Scenario 1 Count:", scenario1_user1.count())
print("User 1 - Scenario 2 Count:", scenario2_user1.count())
print("User 2 - Scenario 1 Count:", scenario1_user2.count())
print("User 2 - Scenario 2 Count:", scenario2_user2.count())

User 1 - Scenario 1 Count: 33513694
User 1 - Scenario 2 Count: 33060379
User 2 - Scenario 1 Count: 33513694
User 2 - Scenario 2 Count: 33060379


# Training ALS Models and Generating Personalized Recommendations

After combining the new user ratings with both recommendation scenarios, ALS collaborative filtering models are trained separately for each case.

The trained ALS models generate top 15 personalized movie recommendations for each user based on viewing preference patterns learned from the MovieLens dataset.

Recommendations are generated for the following four cases:

- User 1 – Scenario 1 (25+ ratings)
- User 1 – Scenario 2 (100+ ratings)
- User 2 – Scenario 1 (25+ ratings)
- User 2 – Scenario 2 (100+ ratings)

The recommendation outputs will later be analyzed to compare how different filtering thresholds influence recommendation quality, diversity, and predicted recommendation confidence.

In [40]:
# Function to train ALS model and display top 15 recommendations

def generate_top_15_recommendations(ratings_RDD, user_id, scenario_name):
    
    print("\n" + scenario_name)
    
    model = ALS.train(
        ratings_RDD,
        best_rank,
        seed=seed,
        iterations=iterations,
        lambda_=regularization_parameter
    )
    
    recommendations = model.recommendProducts(user_id, 15)
    
    recommendation_RDD = sc.parallelize(
        [(rec.product, rec.rating) for rec in recommendations]
    )
    
    recommendations_with_titles = recommendation_RDD.join(
        complete_movies_titles
    ).map(
        lambda x: (x[1][1], x[1][0])
    ).sortBy(
        lambda x: -x[1]
    )
    
    for title, predicted_rating in recommendations_with_titles.collect():
        print(title, " | Predicted Rating:", round(predicted_rating, 3))

In [41]:
# Define final ALS model parameters

best_rank = 8

seed = 5

iterations = 10

regularization_parameter = 0.1

In [43]:
# Create movie ID and title mapping RDD

complete_movies_titles = movies.rdd.map(
    lambda row: (row.movieId, row.title)
)

In [44]:
# User 1 - Scenario 1

generate_top_15_recommendations(
    scenario1_user1,
    999999,
    "User 1 - Scenario 1: Movies with 25+ Ratings"
)


User 1 - Scenario 1: Movies with 25+ Ratings
Band of Brothers (2001)  | Predicted Rating: 5.188
Planet Earth (2006)  | Predicted Rating: 5.166
Planet Earth II (2016)  | Predicted Rating: 5.166
Shawshank Redemption, The (1994)  | Predicted Rating: 5.161
Blue Planet II (2017)  | Predicted Rating: 5.026
Connections (1978)  | Predicted Rating: 5.018
Cosmos: A Spacetime Odissey  | Predicted Rating: 5.001
Schindler's List (1993)  | Predicted Rating: 4.998
The Blue Planet (2001)  | Predicted Rating: 4.996
His Last Vow  | Predicted Rating: 4.96
Cosmos  | Predicted Rating: 4.949
Twelve Angry Men (1954)  | Predicted Rating: 4.934
Civil War, The (1990)  | Predicted Rating: 4.933
Rowan Atkinson: Not Just a Pretty Face (1992)  | Predicted Rating: 4.928
Spider-Man: Across the Spider-Verse (2023)  | Predicted Rating: 4.915


## User 1 – Scenario 1 Recommendation Summary

Using the FULL MovieLens dataset filtered to include movies with at least 25 ratings, the ALS collaborative filtering model generated top 15 personalized movie recommendations for User 1.

The recommendations strongly reflect User 1’s interest in action, science fiction, drama, adventure, and critically acclaimed movies. Several recommendations such as *Band of Brothers*, *Shawshank Redemption*, *Schindler’s List*, and *Spider-Man: Across the Spider-Verse* indicate that the model successfully identified viewing patterns similar to User 1’s original movie preferences.

Many recommended movies received predicted ratings close to or above 5.0, suggesting strong recommendation confidence from the ALS recommendation model.

Scenario 1 produced a diverse set of recommendations because the dataset included movies with at least 25 ratings, allowing the recommendation engine to consider a broader range of movies, including niche and highly rated content.

# User 1 – Scenario 2 Recommendations

This scenario uses the FULL MovieLens dataset filtered to include movies with at least 100 ratings before training the ALS collaborative filtering model.

In [45]:
# User 1 - Scenario 2

generate_top_15_recommendations(
    scenario2_user1,
    999999,
    "User 1 - Scenario 2: Movies with 100+ Ratings"
)


User 1 - Scenario 2: Movies with 100+ Ratings
Band of Brothers (2001)  | Predicted Rating: 5.198
Shawshank Redemption, The (1994)  | Predicted Rating: 5.18
Planet Earth (2006)  | Predicted Rating: 5.168
Planet Earth II (2016)  | Predicted Rating: 5.162
Schindler's List (1993)  | Predicted Rating: 5.028
Blue Planet II (2017)  | Predicted Rating: 5.024
The Blue Planet (2001)  | Predicted Rating: 5.001
Cosmos: A Spacetime Odissey  | Predicted Rating: 4.994
Civil War, The (1990)  | Predicted Rating: 4.985
Cosmos  | Predicted Rating: 4.958
Twelve Angry Men (1954)  | Predicted Rating: 4.952
12 Angry Men (1957)  | Predicted Rating: 4.927
Godfather, The (1972)  | Predicted Rating: 4.917
Human Planet (2011)  | Predicted Rating: 4.909
The Rescue (2021)  | Predicted Rating: 4.905


## User 1 – Scenario 2 Recommendation Summary

Using the FULL MovieLens dataset filtered to include movies with at least 100 ratings, the ALS collaborative filtering model generated top 15 personalized movie recommendations for User 1.

The recommendations continue to reflect User 1’s interest in action, science fiction, drama, and critically acclaimed movies. However, compared to Scenario 1, the recommendations in Scenario 2 are more focused on highly popular and widely rated movies because the filtering threshold removed less frequently rated movies from the dataset.

Movies such as *The Godfather*, *12 Angry Men*, *Planet Earth*, and *Shawshank Redemption* demonstrate that the ALS model identified strong preference similarities between User 1 and other users with similar viewing patterns.

Most predicted recommendation scores remain close to or above 5.0, indicating strong recommendation confidence. The stricter filtering threshold in Scenario 2 produced more stable and mainstream recommendations while reducing recommendation diversity compared to Scenario 1.

In [46]:
# User 2 - Scenario 1

generate_top_15_recommendations(
    scenario1_user2,
    999998,
    "User 2 - Scenario 1: Movies with 25+ Ratings"
)


User 2 - Scenario 1: Movies with 25+ Ratings
Connections (1978)  | Predicted Rating: 5.301
Rowan Atkinson: Not Just a Pretty Face (1992)  | Predicted Rating: 4.999
Planet Earth (2006)  | Predicted Rating: 4.982
Planet Earth II (2016)  | Predicted Rating: 4.977
Spider-Man: Across the Spider-Verse (2023)  | Predicted Rating: 4.961
Gurren Lagann: The Lights in the Sky are Stars (Gekijô ban Tengen toppa guren ragan: Ragan hen) (2009)  | Predicted Rating: 4.955
Us Again (2021)  | Predicted Rating: 4.953
Wild China (2008)  | Predicted Rating: 4.922
Mission Blue (2014)  | Predicted Rating: 4.922
Long Way Round (2004)  | Predicted Rating: 4.871
Piper (2016)  | Predicted Rating: 4.859
Blue Planet II (2017)  | Predicted Rating: 4.859
At the Heart of Gold: Inside the USA Gymnastics Scandal (2019)  | Predicted Rating: 4.846
Prohibition (2011)  | Predicted Rating: 4.833
The Blue Planet (2001)  | Predicted Rating: 4.831


## User 2 – Scenario 1 Recommendation Summary

Using the FULL MovieLens dataset filtered to include movies with at least 25 ratings, the ALS collaborative filtering model generated top 15 personalized movie recommendations for User 2.

The recommendations strongly reflect User 2’s interest in animation, fantasy, adventure, family, and documentary-style movies. Several recommendations such as *Spider-Man: Across the Spider-Verse*, *Piper*, *Planet Earth*, and *Blue Planet II* indicate that the ALS model successfully captured User 2’s preference for visually engaging and highly rated storytelling content.

Compared to User 1, the recommendations for User 2 are more diverse and include a wider mix of animation, international content, documentaries, and niche titles. This demonstrates that the collaborative filtering model adapts recommendations based on individual user preference patterns.

Most predicted recommendation scores remain very high, suggesting strong recommendation confidence from the ALS model. Because Scenario 1 includes movies with at least 25 ratings, the recommendation engine was able to recommend a broader and more diverse range of movies.

In [47]:
# User 2 - Scenario 2

generate_top_15_recommendations(
    scenario2_user2,
    999998,
    "User 2 - Scenario 2: Movies with 100+ Ratings"
)


User 2 - Scenario 2: Movies with 100+ Ratings
Planet Earth (2006)  | Predicted Rating: 4.959
Planet Earth II (2016)  | Predicted Rating: 4.947
Spider-Man: Across the Spider-Verse (2023)  | Predicted Rating: 4.93
Wild China (2008)  | Predicted Rating: 4.914
Blue Planet II (2017)  | Predicted Rating: 4.844
Piper (2016)  | Predicted Rating: 4.837
Band of Brothers (2001)  | Predicted Rating: 4.825
Shawshank Redemption, The (1994)  | Predicted Rating: 4.82
Rabbit Seasoning (1952)  | Predicted Rating: 4.812
Spider-Man: Into the Spider-Verse (2018)  | Predicted Rating: 4.801
The Blue Planet (2001)  | Predicted Rating: 4.794
North & South (2004)  | Predicted Rating: 4.788
John Mulaney: New In Town (2012)  | Predicted Rating: 4.784
Hamilton (2020)  | Predicted Rating: 4.778
Cosmos: A Spacetime Odissey  | Predicted Rating: 4.769


## User 2 – Scenario 2 Recommendation Summary

Using the FULL MovieLens dataset filtered to include movies with at least 100 ratings, the ALS collaborative filtering model generated top 15 personalized movie recommendations for User 2.

The recommendations continue to reflect User 2’s interest in animation, fantasy, family, adventure, and documentary content. Movies such as *Spider-Man: Across the Spider-Verse*, *Planet Earth*, *Blue Planet II*, and *Piper* demonstrate that the ALS model consistently identified strong preference similarities between User 2 and other users with related viewing behaviors.

Compared to Scenario 1, Scenario 2 recommendations are more focused on mainstream and highly rated movies because the stricter filtering threshold removes less frequently rated titles from the recommendation pool. This results in more stable and widely popular recommendations while slightly reducing recommendation diversity.

Several movies appeared in both scenarios because highly rated and strongly matched movies continue to receive high recommendation scores regardless of the filtering threshold. This demonstrates consistency and reliability in the ALS collaborative filtering model.

# Scenario Comparison and Analysis

The recommendation results from all four cases demonstrate how collaborative filtering recommendations change depending on both user preference profiles and dataset filtering thresholds.

## Comparison Between User 1 and User 2

The ALS collaborative filtering model successfully generated different recommendation patterns for the two users based on their movie preferences.

### User 1 Recommendations
User 1 primarily preferred action, science fiction, drama, and adventure movies. As a result, the recommendation engine generated movies such as *Band of Brothers*, *Shawshank Redemption*, *Schindler’s List*, *Spider-Man: Across the Spider-Verse*, and *The Godfather*. Many recommendations focused on critically acclaimed drama, action, and documentary-style content.

### User 2 Recommendations
User 2 primarily preferred animation, fantasy, family, adventure, and emotionally engaging movies. Therefore, the recommendation engine produced recommendations such as *Piper*, *Spider-Man: Into the Spider-Verse*, *Planet Earth*, *Blue Planet II*, and *Rabbit Seasoning*. Compared to User 1, User 2 received more animation, documentary, and family-oriented recommendations.

This demonstrates that the ALS collaborative filtering model effectively adapts recommendations according to individual user preference patterns.

## Comparison Between Scenario 1 and Scenario 2

The two scenarios used different movie filtering thresholds before training the recommendation models.

### Scenario 1 (25+ Ratings)
Scenario 1 included movies with at least 25 ratings. Because the filtering threshold was lower, the recommendation engine had access to a larger and more diverse recommendation pool. This allowed the model to recommend several niche, less mainstream, and specialized movies.

As a result:
- Recommendations were more diverse
- More unique and lesser-known titles appeared
- Recommendation variety increased

### Scenario 2 (100+ Ratings)
Scenario 2 included only movies with at least 100 ratings. This stricter filtering threshold reduced the number of available movies and focused the recommendation engine on more popular and widely rated content.

As a result:
- Recommendations became more mainstream
- Popular movies appeared more frequently
- Recommendation stability and confidence improved
- Diversity slightly decreased compared to Scenario 1


## Overall Insights

Several movies appeared repeatedly across both scenarios because highly rated and strongly matched movies consistently received high recommendation scores from the ALS model. This indicates that the collaborative filtering model successfully identified stable preference similarities between users.

The results demonstrate that:
- Lower filtering thresholds improve recommendation diversity
- Higher filtering thresholds improve recommendation reliability and popularity
- Collaborative filtering can effectively personalize recommendations based on user behavior patterns

Overall, the ALS recommendation engine successfully generated meaningful personalized movie recommendations and demonstrated strong potential for large-scale recommendation systems such as Pumpkinmeter.

# Insights and Foresights

The collaborative filtering recommendation system produced several important insights regarding user behavior, recommendation quality, and the impact of dataset filtering thresholds.

## Key Insights

### Personalized Recommendations Improve User Experience
The ALS collaborative filtering model successfully generated different recommendation lists for User 1 and User 2 based on their individual movie preferences. This demonstrates that recommendation systems can effectively personalize content for different users by identifying similar viewing patterns among large groups of users.

For example:
- User 1 received more action, science fiction, drama, and critically acclaimed movies.
- User 2 received more animation, fantasy, family, and documentary-related recommendations.

This level of personalization can improve customer satisfaction and increase user engagement on a movie streaming or review platform such as Pumpkinmeter.


### Filtering Thresholds Affect Recommendation Diversity
The two recommendation scenarios produced noticeable differences in recommendation behavior.

- Scenario 1 (25+ ratings) generated more diverse and niche recommendations because more movies were available in the recommendation pool.
- Scenario 2 (100+ ratings) generated more stable and mainstream recommendations by focusing on highly rated and frequently reviewed movies.

This demonstrates that filtering thresholds directly affect:
- recommendation diversity
- recommendation reliability
- recommendation popularity
- recommendation confidence


### Highly Rated Movies Consistently Appear Across Scenarios
Several movies appeared repeatedly across multiple scenarios because they received very strong recommendation scores from the ALS model. This indicates that collaborative filtering models can consistently identify highly compatible content for users even when filtering conditions change.

This consistency improves trust in the recommendation engine and suggests that the ALS model learned stable preference relationships from the dataset.


## Business Foresights

### Increased Customer Retention
Personalized recommendations can encourage users to spend more time on the platform by continuously suggesting relevant content. This can improve customer satisfaction and reduce the likelihood of users switching to competing streaming services.

For Pumpkinmeter, an effective recommendation engine could increase:
- customer engagement
- repeat platform visits
- subscription retention
- watch time


### Better Content Discovery
The recommendation engine helps users discover movies they may not have searched for manually. Scenario 1 especially demonstrated how collaborative filtering can expose users to niche and less mainstream content while still matching their interests.

This can improve overall user experience and increase interaction with a wider range of movies available on the platform.


### Scalable Recommendation System
The ALS collaborative filtering model successfully handled the FULL MovieLens dataset containing millions of ratings. This demonstrates that Apache Spark MLlib can efficiently support large-scale recommendation systems for real-world streaming and review platforms.

As Pumpkinmeter grows, Spark’s distributed processing capabilities can support:
- larger user bases
- larger movie catalogs
- real-time recommendation updates
- scalable recommendation generation


### Future Improvements
Although the ALS recommendation system performed effectively, additional improvements could further enhance recommendation quality in future implementations.

Potential future enhancements include:
- incorporating movie genres and tags into recommendations
- using implicit feedback such as watch history and click behavior
- implementing hybrid recommendation systems
- adding real-time recommendation updates
- using deep learning recommendation models for advanced personalization

These enhancements could further improve recommendation accuracy, recommendation diversity, and overall user engagement.

# Business Interpretation

The results of the ALS collaborative filtering recommendation system demonstrate strong business potential for Pumpkinmeter as a personalized movie recommendation platform.

The recommendation engine successfully identified individual user preferences and generated highly personalized movie suggestions using large-scale movie rating data. This shows that collaborative filtering can improve customer experience by helping users quickly discover movies aligned with their interests.

For Pumpkinmeter, personalized recommendations can provide several important business advantages.

## Improved Customer Engagement

Users are more likely to spend additional time on the platform when they receive personalized and relevant movie recommendations. By continuously recommending movies that match individual viewing preferences, the recommendation engine can increase:
- user activity
- watch time
- platform interaction
- repeat visits

The recommendation outputs showed that the ALS model effectively adapted recommendations for different user profiles, which is essential for maintaining long-term customer engagement.


## Increased Customer Retention

Streaming and review platforms face strong competition, and users can easily switch to alternative services if content discovery becomes difficult or repetitive.

The recommendation engine can reduce this risk by improving user satisfaction and continuously introducing users to movies they are likely to enjoy. This personalized experience can strengthen customer loyalty and increase subscription retention for Pumpkinmeter.

## Better Content Discovery

The recommendation system enables users to discover movies beyond standard trending or popular content. Scenario 1 demonstrated that lower filtering thresholds allow more niche and less mainstream movies to appear in recommendations, increasing recommendation diversity.

This capability is valuable because it:
- exposes users to a broader movie catalog
- improves content visibility
- increases interaction with less popular movies
- enhances overall user experience


## Scalability for Large-Scale Platforms

The ALS collaborative filtering model was successfully implemented using Apache Spark MLlib on the FULL MovieLens dataset containing millions of ratings. This demonstrates that Spark’s distributed computing capabilities can efficiently support large-scale recommendation systems.

As Pumpkinmeter expands, the recommendation engine can scale to:
- millions of users
- larger movie catalogs
- increasing recommendation requests
- real-time recommendation generation

This scalability makes Apache Spark a strong technological foundation for enterprise-level recommendation systems.


## Strategic Value for Pumpkinmeter

The recommendation engine can become a core competitive advantage for Pumpkinmeter by delivering:
- personalized user experiences
- improved recommendation accuracy
- higher customer satisfaction
- stronger customer retention
- increased platform engagement

The project results demonstrate that collaborative filtering using ALS can effectively support data-driven decision-making and customer-focused recommendation strategies for modern streaming and movie review platforms.

# Conclusion

This project successfully implemented a collaborative filtering movie recommendation system using Apache Spark MLlib and the Alternating Least Squares (ALS) algorithm on the FULL MovieLens dataset.

The recommendation engine generated personalized movie recommendations for two different users under two separate filtering scenarios:
- Scenario 1: Movies with at least 25 ratings
- Scenario 2: Movies with at least 100 ratings

The results demonstrated that the ALS collaborative filtering model effectively learned user preference patterns and generated meaningful personalized recommendations based on viewing similarities between users.

The project also showed how filtering thresholds influence recommendation behavior:
- Lower filtering thresholds increased recommendation diversity and introduced more niche movies.
- Higher filtering thresholds produced more stable and mainstream recommendations by focusing on widely rated movies.

The recommendation outputs consistently aligned with each user’s movie interests, demonstrating the effectiveness of collaborative filtering for personalized recommendation systems.

From a business perspective, the project highlighted the strong potential of recommendation engines to improve:
- customer engagement
- content discovery
- user satisfaction
- customer retention
- platform scalability

The successful implementation on a large-scale dataset further demonstrated Apache Spark’s ability to efficiently process and analyze millions of movie ratings for enterprise-level recommendation systems.

Overall, the Pumpkinmeter recommendation engine demonstrated how collaborative filtering and big data analytics can support data-driven personalization strategies for modern streaming and movie review platforms.